# Quasineutral no-displacement hot-ion plasma dispersion relation

This notebook solves the homogeneous kinetic dispersion relation for fully kinetic Maxwellian ions and drift-kinetic electrons in a strong guide field $ \mathbf B_0=B_0\hat{\mathbf z} $. The displacement current is fully neglected and Gauss's law is replaced by quasineutrality.

In [ ]:
using Pkg
Pkg.add(["SpecialFunctions", "NLsolve", "CairoMakie"])

using LinearAlgebra
using SpecialFunctions
using NLsolve
using CairoMakie

## Derivation and assumptions

We assume

$$
\mathbf B_0=B_0\hat{\mathbf z},\qquad \mathbf k=(k_\perp,0,k_\parallel)
$$

Ions are fully kinetic Vlasov ions. Electrons are drift kinetic in the strong guide field. The displacement current is neglected, but Gauss's law is replaced by quasineutrality:

$$
\sum_s q_s\delta n_s=0
$$

The retained Maxwell equations are

$$
\nabla\times\mathbf B=\mu_0\mathbf j,\qquad \partial_t\mathbf B=-\nabla\times\mathbf E,\qquad \nabla\cdot\mathbf B=0
$$

Since $ \nabla\cdot\nabla\times\mathbf B=0 $, Ampère's law implies

$$
\nabla\cdot\mathbf j=0
$$

For perturbations $ \propto e^{i(\mathbf k\cdot\mathbf x-\omega t)} $, this gives

$$
\mathbf k\cdot\mathbf j=0
$$

Taking $ \partial_t $ of Faraday's law and using Ampère's law gives

$$
\nabla\times\nabla\times\mathbf E=-\mu_0\partial_t\mathbf j
$$

With the linear kinetic response

$$
\delta\mathbf j=\boldsymbol\sigma\cdot\mathbf E,\qquad \boldsymbol\chi=\frac{i\boldsymbol\sigma}{\varepsilon_0\omega}
$$

we obtain

$$
\mathbf k\times(\mathbf k\times\mathbf E)+\frac{\omega^2}{c^2}\boldsymbol\chi\cdot\mathbf E=0
$$

Using $ \mathbf k\times(\mathbf k\times\mathbf E)=-k^2\mathbf P_T\mathbf E $, with

$$
\mathbf P_T=\mathbf I-\hat{\mathbf k}\hat{\mathbf k}
$$

the dispersion matrix is

$$
\mathbf D(\omega,\mathbf k)=\boldsymbol\chi-\frac{c^2k^2}{\omega^2}\mathbf P_T
$$

and the dispersion relation is

$$
\det\mathbf D(\omega,\mathbf k)=0
$$

The normalization is

$$
\omega\rightarrow\frac{\omega}{\Omega_i},\qquad \mathbf k\rightarrow\rho_i\mathbf k,\qquad \mathbf v\rightarrow\frac{\mathbf v}{v_{th}}
$$

with

$$
v_{th}=\sqrt{\frac{T_i}{m_i}},\qquad \rho_i=\frac{v_{th}}{\Omega_i}
$$

The normalized light speed is

$$
c=\frac{c_{\rm phys}}{v_{th}}
$$

and the ion beta is

$$
\beta_i=\frac{2v_{th}^2}{v_A^2}
$$

Therefore

$$
\omega_{pi}=c\sqrt{\frac{\beta_i}{2}}
$$

Let

$$
\mu=\frac{m_i}{m_e}
$$

Then

$$
\omega_{pe}=\sqrt{\mu}\,\omega_{pi},\qquad v_{te}=\sqrt{\mu\frac{T_e}{T_i}}
$$

In normalized units,

$$
\Omega_i=1,\qquad v_{th}=1,\qquad \rho_i=1
$$

The Maxwellian ion susceptibility uses

$$
b=\frac{k_\perp^2}{2},\qquad \zeta_n=\frac{\omega-n}{\sqrt{2}\,k_\parallel},\qquad \zeta_0=\frac{\omega}{\sqrt{2}\,k_\parallel}
$$

and

$$
\Gamma_n(b)=e^{-b}I_{|n|}(b)
$$

The full ion tensor is

$$
\boldsymbol\chi_i=\frac{\omega_{pi}^2}{\omega^2}\sum_{n=-\infty}^{\infty}\mathbf X_n
$$

with entries

$$
X_{n,xx}=\frac{n^2}{b}\Gamma_n\zeta_0Z(\zeta_n)
$$

$$
X_{n,xy}=in\Gamma_n'\zeta_0Z(\zeta_n),\qquad X_{n,yx}=-X_{n,xy}
$$

$$
X_{n,yy}=\left(\frac{n^2}{b}\Gamma_n-2b\Gamma_n'\right)\zeta_0Z(\zeta_n)
$$

$$
X_{n,xz}=X_{n,zx}=-\frac{n}{\sqrt{2b}}\Gamma_n\zeta_0Z'(\zeta_n)
$$

$$
X_{n,yz}=i\sqrt{\frac{b}{2}}\Gamma_n'\zeta_0Z'(\zeta_n),\qquad X_{n,zy}=-X_{n,yz}
$$

$$
X_{n,zz}=-\Gamma_n\zeta_0\zeta_nZ'(\zeta_n)
$$

For exactly $ k_\parallel=0 $, we analytically take the large-$ \zeta $ limit:

$$
\zeta_0Z(\zeta_n)\rightarrow-\frac{\omega}{\omega-n},\qquad \zeta_0Z'(\zeta_n)\rightarrow0
$$

This avoids numerical cancellation in $ Z(\zeta) $.

The drift-kinetic electron susceptibility is

$$
\boldsymbol\chi_e^{\rm DK}=\begin{pmatrix}0&-i\chi_H&0\\ i\chi_H&0&0\\0&0&\chi_{e\parallel}\end{pmatrix}
$$

with

$$
\chi_H=\frac{\omega_{pi}^2}{\omega}
$$

and

$$
\chi_{e\parallel}=-\frac{\omega_{pe}^2}{\omega^2}\zeta_e^2Z'(\zeta_e),\qquad \zeta_e=\frac{\omega}{\sqrt{2}\,k_\parallel v_{te}}
$$

For $ k_\parallel=0 $,

$$
\chi_{e\parallel}\rightarrow-\frac{\omega_{pe}^2}{\omega^2}
$$

In [ ]:
const βi = 0.1
const μ = 10.0
const Te_Ti = 1.0
const Nharm = 12

const Ωi = 1.0
const vth = 1.0
const c = 1000.0

const ωpi = c*sqrt(βi/2)
const ωpe = sqrt(μ)*ωpi
const vte = sqrt(μ*Te_Ti)
const βe = Te_Ti*βi

const di_over_ρi = c/ωpi
const de_over_ρi = c/ωpe
const kpar_tol = 1e-12

println("βi       = ", βi)
println("μ        = ", μ)
println("Te/Ti    = ", Te_Ti)
println("c/vth    = ", c)
println("ωpi/Ωi   = ", ωpi)
println("ωpe/Ωi   = ", ωpe)
println("vte/vth  = ", vte)
println("βe       = ", βe)
println("di/ρi    = ", di_over_ρi)
println("de/ρi    = ", de_over_ρi)

In [ ]:
function Zfun(z::ComplexF64)
    if abs(z) > 30
        return -1/z - 1/(2z^3) - 3/(4z^5) - 15/(8z^7)
    else
        return im*sqrt(π)*erfcx(-im*z)
    end
end

function Zprime(z::ComplexF64)
    if abs(z) > 30
        return 1/z^2 + 3/(2z^4) + 15/(4z^6) + 105/(8z^8)
    else
        return -2*(1 + z*Zfun(z))
    end
end

Γ(n::Integer, b::Real) = exp(-b)*besseli(abs(n), b)

function Γprime(n::Integer, b::Real)
    m = abs(n)
    return exp(-b)*(0.5*(besseli(abs(m-1), b) + besseli(m+1, b)) - besseli(m, b))
end

function scaled_det(D::AbstractMatrix{<:Complex})
    A = Matrix{ComplexF64}(D)
    for i in axes(A,1)
        s = norm(view(A,i,:))
        s > 0 && (A[i,:] ./= s)
    end
    for j in axes(A,2)
        s = norm(view(A,:,j))
        s > 0 && (A[:,j] ./= s)
    end
    return det(A)
end

function singular_ratio(D::AbstractMatrix{<:Complex})
    s = svdvals(Matrix{ComplexF64}(D))
    return minimum(s)/maximum(s)
end

In [ ]:
function ion_susceptibility(ω, kperp, kpar)
    ω = ComplexF64(ω)
    @assert abs(kpar) > kpar_tol "Use ion_susceptibility_kpar0 for k_parallel = 0."

    b = max(kperp^2/2, 1e-14)
    ζ0 = ω/(sqrt(2)*kpar)
    χ = zeros(ComplexF64, 3, 3)

    for n in -Nharm:Nharm
        ζn = (ω - n)/(sqrt(2)*kpar)
        Zn = Zfun(ζn)
        Zpn = Zprime(ζn)
        Gn = Γ(n, b)
        Gpn = Γprime(n, b)
        A = ζ0*Zn

        χ[1,1] += (n^2/b)*Gn*A
        χ[1,2] += im*n*Gpn*A
        χ[2,1] -= im*n*Gpn*A
        χ[2,2] += ((n^2/b)*Gn - 2b*Gpn)*A

        χ[1,3] -= (n/sqrt(2b))*Gn*ζ0*Zpn
        χ[3,1] -= (n/sqrt(2b))*Gn*ζ0*Zpn
        χ[2,3] += im*sqrt(b/2)*Gpn*ζ0*Zpn
        χ[3,2] -= im*sqrt(b/2)*Gpn*ζ0*Zpn
        χ[3,3] -= Gn*ζ0*ζn*Zpn
    end

    return (ωpi^2/ω^2)*χ
end

function ion_susceptibility_kpar0(ω, kperp)
    ω = ComplexF64(ω)
    b = kperp^2/2
    χ = zeros(ComplexF64, 3, 3)

    if b < 1e-12
        den = ω^2 - 1
        χ[1,1] = -ωpi^2/den
        χ[2,2] = χ[1,1]
        χ[1,2] = -im*ωpi^2/(ω*den)
        χ[2,1] = -χ[1,2]
        χ[3,3] = -ωpi^2/ω^2
        return χ
    end

    for n in -Nharm:Nharm
        Δn = ω - n
        Gn = Γ(n, b)
        Gpn = Γprime(n, b)
        A = -ω/Δn

        χ[1,1] += (n^2/b)*Gn*A
        χ[1,2] += im*n*Gpn*A
        χ[2,1] -= im*n*Gpn*A
        χ[2,2] += ((n^2/b)*Gn - 2b*Gpn)*A
        χ[3,3] += -Gn*ω/Δn
    end

    return (ωpi^2/ω^2)*χ
end

function electron_DK_susceptibility(ω, kpar)
    ω = ComplexF64(ω)
    χe = zeros(ComplexF64, 3, 3)

    χH = ωpi^2/ω
    χe[1,2] = -im*χH
    χe[2,1] = im*χH

    if abs(kpar) <= kpar_tol
        χe[3,3] = -ωpe^2/ω^2
    else
        ζe = ω/(sqrt(2)*kpar*vte)
        χe[3,3] = -(ωpe^2/ω^2)*ζe^2*Zprime(ζe)
    end

    return χe
end

function total_susceptibility(ω, kx, kz)
    χi = abs(kz) <= kpar_tol ? ion_susceptibility_kpar0(ω, abs(kx)) : ion_susceptibility(ω, abs(kx), kz)
    χe = electron_DK_susceptibility(ω, kz)
    return χi + χe
end

function projectors(kx, kz)
    kv = [kx, 0.0, kz]
    k2 = dot(kv, kv)
    @assert k2 > 0
    PL = ComplexF64.(kv*kv'/k2)
    PT = Matrix{ComplexF64}(I, 3, 3) - PL
    return PL, PT, k2
end

function qn_ndc_matrix(ω, kx, kz)
    ω = ComplexF64(ω)
    PL, PT, k2 = projectors(kx, kz)
    χ = total_susceptibility(ω, kx, kz)
    return χ - (c^2*k2/ω^2)*PT
end

dispersion(ω, kx, kz) = det(qn_ndc_matrix(ω, kx, kz))

## Cold reference

The cold reference uses cold ions, the same massless-electron Hall current, and cold parallel ion/electron inertia. It is assembled with the same quasineutral no-displacement operator.

In [ ]:
function cold_hybrid_susceptibility(ω)
    ω = ComplexF64(ω)
    deni = ω^2 - 1

    χS_i = -ωpi^2/deni
    χD_i = ωpi^2/(ω*deni)
    χD_e = ωpi^2/ω
    χD = χD_i + χD_e

    χ = zeros(ComplexF64, 3, 3)
    χ[1,1] = χS_i
    χ[2,2] = χS_i
    χ[1,2] = -im*χD
    χ[2,1] = im*χD
    χ[3,3] = -(ωpi^2 + ωpe^2)/ω^2
    return χ
end

function cold_qn_ndc_matrix(ω, kx, kz)
    ω = ComplexF64(ω)
    PL, PT, k2 = projectors(kx, kz)
    return cold_hybrid_susceptibility(ω) - (c^2*k2/ω^2)*PT
end

## Root finders

In [ ]:
function solve_seed(Dmat, kx, kz, seed; ftol=1e-10)
    function residual!(F, x)
        ω = complex(x[1], x[2])
        d = scaled_det(Dmat(ω, kx, kz))
        F[1] = real(d)
        F[2] = imag(d)
    end

    try
        sol = nlsolve(residual!, [real(seed), imag(seed)]; method=:trust_region, ftol=ftol, xtol=1e-10, iterations=300)
        return converged(sol) ? complex(sol.zero[1], sol.zero[2]) : nothing
    catch
        return nothing
    end
end

function complex_roots_at_k(Dmat, kx, kz; ωmin=0.02, ωmax=8.0, nseed=120, imag_seeds=(0.0, -0.01, -0.05, -0.15, -0.4), srtol=1e-6)
    roots = ComplexF64[]
    for wr in range(ωmin, ωmax, length=nseed), wi in imag_seeds
        ω = solve_seed(Dmat, kx, kz, complex(wr, wi))
        ω === nothing && continue
        if !(ωmin < real(ω) < ωmax && imag(ω) <= 1e-5 && imag(ω) > -3.0)
            continue
        end
        D = Dmat(ω, kx, kz)
        if singular_ratio(D) < srtol && abs(scaled_det(D)) < 1e-6
            if all(abs(ω-r) > 2e-3 for r in roots)
                push!(roots, ω)
            end
        end
    end
    return sort!(roots, by=real)
end

function bisect_real_root(f, a, b; tol=1e-10, maxit=100)
    fa = f(a)
    fb = f(b)
    if !isfinite(fa) || !isfinite(fb) || fa*fb > 0
        return nothing
    end
    for _ in 1:maxit
        m = 0.5*(a+b)
        fm = f(m)
        if !isfinite(fm)
            return nothing
        end
        if abs(fm) < tol || abs(b-a) < tol
            return m
        end
        if fa*fm <= 0
            b = m
            fb = fm
        else
            a = m
            fa = fm
        end
    end
    return 0.5*(a+b)
end

function real_roots_at_k(Dmat, kx, kz; ωmin=0.02, ωmax=8.0, nscan=5000, pole_pad=1e-4, srtol=1e-7)
    f(ωr) = real(scaled_det(Dmat(complex(ωr, 0.0), kx, kz)))
    grid = collect(range(ωmin, ωmax, length=nscan))
    roots = ComplexF64[]
    lastω = NaN
    lastf = NaN

    for ωr in grid
        if minimum(abs.(ωr .- collect(1:Nharm))) < pole_pad
            lastω = NaN
            lastf = NaN
            continue
        end

        fr = try
            f(ωr)
        catch
            NaN
        end

        if !isfinite(fr) || abs(fr) > 1e8
            lastω = NaN
            lastf = NaN
            continue
        end

        if isfinite(lastf) && lastf*fr <= 0
            root = bisect_real_root(f, lastω, ωr)
            if root !== nothing && ωmin < root < ωmax
                ω = complex(root, 0.0)
                D = Dmat(ω, kx, kz)
                if singular_ratio(D) < srtol || abs(scaled_det(D)) < 1e-7
                    if all(abs(ω-r) > 2e-3 for r in roots)
                        push!(roots, ω)
                    end
                end
            end
        end

        lastω = ωr
        lastf = fr
    end

    return sort!(roots, by=real)
end

function scalar_real_roots(f; ωmin=0.02, ωmax=8.0, nscan=5000, pole_pad=1e-4)
    grid = collect(range(ωmin, ωmax, length=nscan))
    roots = Float64[]
    lastω = NaN
    lastf = NaN

    for ωr in grid
        if minimum(abs.(ωr .- collect(1:Nharm))) < pole_pad
            lastω = NaN
            lastf = NaN
            continue
        end

        fr = try
            f(ωr)
        catch
            NaN
        end

        if !isfinite(fr) || abs(fr) > 1e8
            lastω = NaN
            lastf = NaN
            continue
        end

        if isfinite(lastf) && lastf*fr <= 0
            r = bisect_real_root(f, lastω, ωr)
            if r !== nothing && all(abs(r-r0) > 2e-3 for r0 in roots)
                push!(roots, r)
            end
        end

        lastω = ωr
        lastf = fr
    end

    return sort!(roots)
end

function roots_to_matrix(kvals, root_lists)
    nb = maximum(length.(root_lists); init=0)
    M = fill(NaN, length(kvals), nb)
    for (i, rs) in enumerate(root_lists)
        for (j, r) in enumerate(rs)
            M[i,j] = real(r)
        end
    end
    return M
end

## Complex-plane check

In [ ]:
ωr_grid = range(0.02, 3.0, length=220)
ωi_grid = range(-1.0, 0.2, length=180)
kx_cp, kz_cp = 0.0, 0.3

Dplane = [scaled_det(qn_ndc_matrix(complex(wr, wi), kx_cp, kz_cp)) for wr in ωr_grid, wi in ωi_grid]

fig = Figure(size=(800, 420))
ax = Axis(fig[1,1], xlabel="Re(ω)/Ωi", ylabel="Im(ω)/Ωi", title="Complex-plane structure, kx=$(kx_cp), kz=$(kz_cp)")
hm = heatmap!(ax, ωr_grid, ωi_grid, imag.(log.(Dplane) .+ 1e-16); colormap=:phase)
contour!(ax, ωr_grid, ωi_grid, real.(Dplane); levels=[0.0], color=:cyan, linewidth=1.2)
contour!(ax, ωr_grid, ωi_grid, imag.(Dplane); levels=[0.0], color=:red, linewidth=1.2)
hlines!(ax, [0.0]; color=:white, linestyle=:dash, alpha=0.7)
Colorbar(fig[1,2], hm; label="log10 |det Dhat|")
fig

## Cold reference dispersion

In [ ]:
kz_cold = range(0.02, 0.8, length=70)
kx_cold = range(0.05, 4.0, length=90)

cold_par_roots = [real_roots_at_k(cold_qn_ndc_matrix, 0.0, kz; ωmax=8.0, nscan=3000) for kz in kz_cold]
cold_perp_roots = [real_roots_at_k(cold_qn_ndc_matrix, kx, 0.0; ωmax=8.0, nscan=3000) for kx in kx_cold]

cold_par_mat = roots_to_matrix(kz_cold, cold_par_roots)
cold_perp_mat = roots_to_matrix(kx_cold, cold_perp_roots)

fig = Figure(size=(1100, 450))
ax1 = Axis(fig[1,1], xlabel="kz ρi", ylabel="ω/Ωi", title="Parallel cold quasineutral no-displacement dispersion", limits=(nothing, (0,8)))
ax2 = Axis(fig[1,2], xlabel="kx ρi", ylabel="ω/Ωi", title="Perpendicular cold quasineutral no-displacement dispersion", limits=(nothing, (0,8)))

for j in 1:size(cold_par_mat,2)
    lines!(ax1, collect(kz_cold), cold_par_mat[:,j]; linewidth=2)
end

for j in 1:size(cold_perp_mat,2)
    lines!(ax2, collect(kx_cold), cold_perp_mat[:,j]; linewidth=2)
end

fig

## Parallel propagation

In [ ]:
function weakly_damped_roots_at_k(Dmat, kx, kz; ωmin=0.02, ωmax=8.0, nseed=120, γmax=1e-3, srtol=1e-6)
    roots = ComplexF64[]
    imag_seeds = (0.0, -1e-5, -1e-4, -5e-4, -1e-3)

    for wr in range(ωmin, ωmax, length=nseed), wi in imag_seeds
        ω = solve_seed(Dmat, kx, kz, complex(wr, wi))
        ω === nothing && continue

        if !(ωmin < real(ω) < ωmax && abs(imag(ω)) < γmax)
            continue
        end

        D = Dmat(ω, kx, kz)

        if singular_ratio(D) < srtol && abs(scaled_det(D)) < 1e-6
            if all(abs(real(ω) - real(r)) > 2e-3 for r in roots)
                push!(roots, ω)
            end
        end
    end

    return sort!(roots, by=real)
end


kz_values = range(0.02, 3, length=50)
γmax = 5e-1

parallel_spectrum = Tuple{Float64,ComplexF64}[]

for kz in kz_values
    rs = weakly_damped_roots_at_k(qn_ndc_matrix, 0.0, kz;
        ωmax=8.0,
        nseed=140,
        γmax=γmax
    )

    for ω in rs
        push!(parallel_spectrum, (kz, ω))
    end
end

cold_par_ref = roots_to_matrix(
    kz_values,
    [
        real_roots_at_k(cold_qn_ndc_matrix, 0.0, kz;
            ωmax=8.0,
            nscan=3000
        )
        for kz in kz_values
    ]
)

kpar_k = first.(parallel_spectrum)
ωpar_k = last.(parallel_spectrum)

ωre = real.(ωpar_k)
ωim = imag.(ωpar_k)

mask_red  = ωre .< 1.0
mask_blue = ωre .> 1.0
mask_eq   = ωre .== 1.0

fig = Figure(size=(800, 700))

ax1 = Axis(fig[1, 1],
    xlabel="kz ρi",
    ylabel="Re(ω)/Ωi",
    title="Parallel propagation: QN no-displacement (hot vs cold)",
    limits=(nothing, (0, 8))
)

ax2 = Axis(fig[2, 1],
    xlabel="kz ρi",
    ylabel="Im(ω)/Ωi",
    title="Only roots with |Im(ω)| < $(γmax)"
)

for j in 1:size(cold_par_ref, 2)
    lines!(ax1, collect(kz_values), cold_par_ref[:, j];
        color=:black,
        linestyle=:dash,
        linewidth=1.5,
        label=(j == 1 ? "cold" : nothing)
    )
end

scatter!(ax1, kpar_k[mask_red], ωre[mask_red];
    color=:red,
    label="kinetic Re(ω)<1"
)

scatter!(ax1, kpar_k[mask_blue], ωre[mask_blue];
    color=:blue,
    label="kinetic Re(ω)>1"
)

scatter!(ax1, kpar_k[mask_eq], ωre[mask_eq];
    color=:gray,
    label="kinetic Re(ω)=1"
)

axislegend(ax1; position=:lt)

scatter!(ax2, kpar_k[mask_red], ωim[mask_red];
    color=:red
)

scatter!(ax2, kpar_k[mask_blue], ωim[mask_blue];
    color=:blue
)

scatter!(ax2, kpar_k[mask_eq], ωim[mask_eq];
    color=:gray
)

hlines!(ax2, [0.0];
    color=:gray,
    linestyle=:dash
)

fig

## Perpendicular propagation and quasineutral ion Bernstein roots

For $ k_\parallel=0 $, the analytic large-$ \zeta $ expansion is used. In the quasineutral no-displacement model, the longitudinal electrostatic condition is $ \chi_{xx}=0 $, not $ 1+\chi_{xx}=0 $.

In [ ]:
function quasineutral_bernstein_roots(kx; ωmax=8.0)
    f(ωr) = real(total_susceptibility(complex(ωr, 0.0), abs(kx), 0.0)[1,1])
    return scalar_real_roots(f; ωmin=0.02, ωmax=ωmax, nscan=5000)
end

kx_values = range(0.05, 4.0, length=75)

perp_roots = [real_roots_at_k(qn_ndc_matrix, kx, 0.0; ωmax=8.0, nscan=5000, srtol=1e-6) for kx in kx_values]
bern_roots = [quasineutral_bernstein_roots(kx; ωmax=8.0) for kx in kx_values]
cold_perp_ref = roots_to_matrix(kx_values, [real_roots_at_k(cold_qn_ndc_matrix, kx, 0.0; ωmax=8.0, nscan=3000) for kx in kx_values])

perp_spectrum = Tuple{Float64,Float64}[]
bern_spectrum = Tuple{Float64,Float64}[]

for (kx, rs) in zip(kx_values, perp_roots)
    for r in rs
        push!(perp_spectrum, (kx, real(r)))
    end
end

for (kx, rs) in zip(kx_values, bern_roots)
    for r in rs
        push!(bern_spectrum, (kx, r))
    end
end

fig = Figure(size=(850, 650))
ax = Axis(fig[1,1], xlabel="kx ρi", ylabel="ω/Ωi",
          title="Perpendicular propagation: QN no-displacement (hot vs cold)",
          limits=(nothing, (0,8)))

for j in 1:size(cold_perp_ref, 2)
    lines!(ax, collect(kx_values), cold_perp_ref[:,j];
           color=:black, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold" : nothing))
end

scatter!(ax, first.(perp_spectrum), last.(perp_spectrum);
         label="hot (DK electrons)")

axislegend(ax; position=:lt)
fig

## Notes

The dispersion matrix used here is

$$
\mathbf D=\boldsymbol\chi-\frac{c^2k^2}{\omega^2}\mathbf P_T
$$

There is no Poisson term $ \mathbf P_L $. The longitudinal electric field is determined algebraically by quasineutrality/current continuity, not by Gauss's law. Therefore the quasineutral electrostatic Bernstein condition is $ \chi_{LL}=0 $, while the full coupled electromagnetic-quasineutral roots come from $ \det\mathbf D=0 $.

## Comparison with bslLD simulation

We now run the `EMSolverDKNoPol` solver from `bslLD` — the Darwin drift-kinetic solver without polarisation drift — and overlay the simulated $\omega$–$k$ power spectrum with the analytic dispersion curves computed above. The simulation uses the same parameters as the analytic calculation: $\beta_i=0.1$, $\mu=m_i/m_e=10$ ($\mu_e=m_e/m_i=0.1$), $T_e/T_i=1$, $T_i=1$ (kinetic ions). The initial condition is a small Maxwellian perturbation that excites all modes.

Two runs are performed:
- **Parallel** ($k\parallel B_0$): `Bdir=1`, wave vector along $\hat{x}$, background field $B_0\hat{x}$.
- **Perpendicular** ($k\perp B_0$): `Bdir=3`, wave vector along $\hat{x}$, background field $B_0\hat{z}$.

## Numerical method: time stepping and field solver

### Time-stepping algorithm

The notebook's `step!` function implements a symmetric **V–X–F–X–V** Störmer–Verlet split. Each macro-step of size $\Delta t$ proceeds as:

| Sub-step | Action | Evaluation phase |
|---|---|---|
| 1 | $\tfrac{1}{2}\Delta t$ velocity push — `advectV!(f, E^n)` | $\theta = \theta_0 + \tfrac{1}{4}\Omega_i\Delta t$ |
| 2 | $\tfrac{1}{2}\Delta t$ spatial push — `advectX!(f)` | $\theta = \theta_0 + \tfrac{1}{4}\Omega_i\Delta t$ |
| 3 | **Evaluate moments** $n_i$, $\mathbf{J}_\perp$ from the half-advanced $f$ | phase $= \theta_0 + \tfrac{1}{2}\Omega_i\Delta t$ |
| 4 | **Field solve** — advance $(E^n, B^n) \to (E^{n+1}, B^{n+1})$ using $\mathbf{J}_\perp^{n+1/2}$ | — |
| 5 | $\tfrac{1}{2}\Delta t$ spatial push — `advectX!(f)` | $\theta = \theta_0 + \tfrac{3}{4}\Omega_i\Delta t$ |
| 6 | $\tfrac{1}{2}\Delta t$ velocity push — `advectV!(f, E^{n+1})` | $\theta = \theta_0 + \tfrac{3}{4}\Omega_i\Delta t$ |

The splitting is second-order accurate and time-reversible. A 2/3-rule Fourier filter is applied to $E^{n+1}$ after step 4 to suppress the Nyquist pile-up in the velocity-space interpolation. The gyrophase argument $\theta = \theta_0 + \Omega_i t$ is advanced analytically inside each BSL back-trace; only the velocity-space advection depends on the electric field, and the field passed to step 6 is the newly computed $E^{n+1}$.

---

### EMSolverDKNoPol field solver

`EMSolverDKNoPol` (Darwin drift-kinetic, no polarisation drift) advances $(E, B)$ by one full step $\Delta t$ in a **single spectral Cramér solve**. Working mode by mode in Fourier space, the solver simultaneously enforces:

- **Faraday's law** (explicit, centred at $n+\tfrac{1}{2}$):
$$\hat{\mathbf{B}}^{n+1} = \hat{\mathbf{B}}^n - \Delta t\,(i\mathbf{k}\times\hat{\mathbf{E}}^{n+1})$$

- **Ampère's law without displacement current** (NDC), linearised about $B^n$:
$$i\mathbf{k}\times\hat{\mathbf{B}}^n = \tfrac{\beta_i}{2}\hat{\mathbf{J}}_\perp^{n+1/2}$$

Substituting Faraday into the current-balance equation eliminates $\hat{\mathbf{B}}^{n+1}$ and yields a $3\times 3$ real linear system per mode for $\hat{\mathbf{E}}^{n+1}$, solved by Cramér's rule. The background parallel direction (indexed by `pz = grid.Bdir`) is coupled through the polarisation-pressure term $\nabla\cdot\boldsymbol\Pi_{\rm diff}$; in the present notebook `Pi_diff` is set to zero.

**For strictly perpendicular 1D propagation** ($\mathbf{k} = k_x\hat{x}$, $\mathbf{B}_0 = B_0\hat{z}$, `Bdir=3`), the $3\times 3$ system degenerates. Using $c \equiv 2/\beta_i$ (normalised Alfvén-speed squared) and $\alpha \equiv c\,\Delta t$, the Cramér solution reduces to:

$$\hat{E}_x^{n+1} = -c\,ik_x\hat{B}_z^n - \hat{J}_y - \alpha k_x^2\hat{J}_x$$

$$\hat{E}_y^{n+1} = \hat{J}_x$$

$$\hat{B}_z^{n+1} = \hat{B}_z^n - \Delta t\,ik_x\hat{E}_x^{n+1}$$

The longitudinal field $\hat{E}_x$ is sourced by the **transverse magnetic field $\hat{B}_z$** (via Faraday+Ampère) and the **transverse ion current $\hat{J}_y$** — the electromagnetic channel — plus a correction $\alpha k_x^2\hat{J}_x$ proportional to the longitudinal current.

---

### Mismatch with the analytical quasineutral model

The analytical dispersion matrix is $\mathbf{D} = \boldsymbol\chi - (c^2k^2/\omega^2)\mathbf{P}_T$. For $\mathbf{k} = k_x\hat{x}$ the transverse projector has $P_{T,xx} = 0$, so:

$$D_{xx} = \chi_{xx}, \qquad D_{yy} = \chi_{yy} - \frac{c^2k^2}{\omega^2}$$

The full $2\times 2$ block determinant is:

$$\chi_{xx}\!\left(\chi_{yy} - \frac{c^2k^2}{\omega^2}\right) - \chi_{xy}\chi_{yx} = 0$$

This has two families of roots:
- **Shear Alfvén / EM mode**: dominated by the $-c^2k^2/\omega^2$ term.
- **Ion Bernstein wave (IBW)**: lives in the $D_{xx} = \chi_{xx}$ row — the quasineutral electrostatic condition $\chi_{xx} = 0$.

The solver never evaluates $\chi_{xx}$. Instead it computes $\hat{E}_x$ from the EM channel, which implements $\chi_{yx}$ (the off-diagonal response coupling $E_x$ to $J_y$), **not** the electrostatic diagonal $\chi_{xx}$.

| | Longitudinal $E_x$ | IBW condition |
|---|---|---|
| **Analytical (QN NDC)** | $D_{xx} = \chi_{xx}$ — quasineutrality | $\chi_{xx} = 0$ pole |
| **EMSolverDKNoPol** | $\hat{E}_x$ from Faraday/Ampère + $\hat{J}_y$ | no $\chi_{xx} = 0$ pole |

The solver correctly implements NDC for the transverse EM fields $(E_y, B_z)$ but replaces the quasineutral electrostatic condition for $E_x$ with the EM leapfrog equation. These are not equivalent for purely perpendicular propagation: the QN condition gives a pole at $\chi_{xx} = 0$, while the EM leapfrog gives a smooth response proportional to $J_y$. Adding a Poisson or quasineutral step to compute $E_x$ from $\delta n_i$ would close this gap and bring the IBW branches into agreement with the analytical model.

In [ ]:
include("../scripts/select_backend.jl")  # loads bslLD
using Random, FFTW, DSP, Statistics

mutable struct Diag
    Ex::Vector
    Ey::Vector
    Bz::Vector
    t::Vector
end
Diag() = Diag([], [], [], [])

function record!(diag::Diag, sol, simTime)
    push!(diag.Ex, copy(sol.E[1].data))
    push!(diag.Ey, copy(sol.E[2].data))
    push!(diag.Bz, copy(sol.B[3].data))
    push!(diag.t,  simTime.current_T)
end

function step!(f_i, sol, grid, simTime, solver)
    phase_start = simTime.phase
    Ω  = simTime.gyro_frequency
    dt = simTime.dt
    Pi_zero = bslLD.zero_vectorfield3(grid)

    simTime.fraction_dt = 0.5
    simTime.phase = phase_start + 0.25 * Ω * dt
    bslLD.advectV!(f_i, grid, simTime, sol.E)

    simTime.phase = phase_start + 0.25 * Ω * dt
    simTime.fraction_dt = 0.5
    bslLD.advectX!(f_i, grid, simTime)

    simTime.phase = phase_start + 0.5 * Ω * dt
    n_i    = bslLD.compute_density(f_i, grid)
    J_perp = bslLD.compute_current(f_i, grid, simTime.phase)
    moments_mid = bslLD.Moments(n_i, J_perp, Pi_zero)

    bslLD.solve_fields!(sol, moments_mid, grid, solver, dt)

    # for (ind, Ec) in enumerate(sol.Enew)
    #     sol.Enew[ind].data .= bslLD.fourier_filter(Ec, grid, 0.7).data
    # end

    simTime.phase = phase_start + 0.75 * Ω * dt
    simTime.fraction_dt = 0.5
    bslLD.advectX!(f_i, grid, simTime)

    simTime.phase = phase_start + 0.75 * Ω * dt
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f_i, grid, simTime, sol.Enew)

    sol.E .= sol.Enew
    simTime.phase = phase_start
    simTime.fraction_dt = 1.0
end

function make_ics_sim(grid, eps_val, T_val, Nx_sim)
    Ex = eps_val .* randn(Nx_sim)
    Ex .-= mean(Ex)
    Ey = eps_val .* randn(Nx_sim)
    Ey .-= mean(Ey)
    E0 = bslLD.VectorField([
        bslLD.ScalarField(Ex),
        bslLD.ScalarField(Ey),
        bslLD.ScalarField(zeros(Nx_sim)),
    ])
    B0  = bslLD.VectorField([bslLD.ScalarField(zeros(Nx_sim)) for _ in 1:3])
    sol = bslLD.FieldSolution(E0, B0)
    initFuncv(v) = exp(-(v + eps_val*rand())^2 / (2T_val)) / sqrt(2π*T_val)
    f_i = bslLD.Distribution(grid, 0.0, initFuncx = x -> 1.0, initFuncv = initFuncv)
    return f_i, sol
end

function make_spec_sim(diag, dt_val, grid_sim, Nx_sim; field = :Bz)
    data_src = field == :Ex ? diag.Ex : field == :Ey ? diag.Ey : diag.Bz
    omega = fftfreq(length(diag.t), 1/dt_val) .* 2π
    k     = fftfreq(Nx_sim, 1/grid_sim.delta[1]) .* 2π
    nw    = length(diag.t) ÷ 2
    nk    = Nx_sim ÷ 2
    w_win = kaiser(length(diag.t), 3)
    data  = transpose(hcat(data_src...))
    spec  = log.(abs.(fft(data .* w_win))[1:nw, 1:nk] .+ 1e-30)
    return k, omega, spec, nk, nw
end

function run_sim(solver_type, mu_em, dt_val, grid_sim, T_val, Tmax_val, eps_val, Nx_sim)
    Random.seed!(42)
    f_i, sol  = make_ics_sim(grid_sim, eps_val, T_val, Nx_sim)
    solver_fi = solver_type(βi, mu_em)
    simTime   = bslLD.SimulationTime(dt_val, Tmax_val)
    bslLD.ProgressMeter.ijulia_behavior(:clear)
    diag = Diag()
    while bslLD.continue_advection(simTime, true)
        step!(f_i, sol, grid_sim, simTime, solver_fi)
        bslLD.advance!(simTime)
        record!(diag, sol, simTime)
    end
    return diag
end

function run_sim_cold(solver_type, mu_em, dt_val, grid_sim, Tmax_val, eps_val, Nx_sim)
    Random.seed!(42)
    _, sol    = make_ics_sim(grid_sim, eps_val, 1.0, Nx_sim)
    fluid     = bslLD.ColdIonFluid(grid_sim)
    solver_fi = solver_type(βi, mu_em)
    simTime   = bslLD.SimulationTime(dt_val, Tmax_val)
    bslLD.ProgressMeter.ijulia_behavior(:clear)
    diag = Diag()
    while bslLD.continue_advection(simTime, true)
        bslLD.step_cold!(fluid, sol, grid_sim, simTime, solver_fi)
        bslLD.advance!(simTime)
        record!(diag, sol, simTime)
    end
    return diag
end


In [ ]:
# Simulation parameters (matching the analytic run above)
const sim_mu_em = 1/μ       # mₑ/mᵢ = 0.1 (bslLD convention)
const sim_T     = 1.0       # kinetic ions
const sim_eps   = 1e-6
const sim_dt    = 0.005
const sim_Tmax  = 100.0
const sim_Lx    = 20π
const sim_Nx    = 64
const sim_Nv    = 16
const sim_vmax  = 4.0 * sqrt(sim_T)

println("mu_em (mₑ/mᵢ) = ", sim_mu_em)
println("T_i           = ", sim_T)
println("dt            = ", sim_dt, "  Tmax = ", sim_Tmax)
println("Lx/ρi         = ", round(sim_Lx; digits=3),
        "  kmin = ", round(2π/sim_Lx; digits=4),
        "  kmax = ", round(2π*(sim_Nx÷2)/sim_Lx; digits=3))

# Cold simulation uses a smaller dt and longer Tmax
# (step_cold! is cheap since there is no kinetic advection)
const sim_dt_cold   = 0.005
const sim_Tmax_cold = 100.0
const sim_eps_cold  = 1e-7

In [ ]:
# Bdir=1: B₀ ∥ x̂ = wave direction → k ∥ B₀ (parallel propagation)
grid_par = bslLD.Grid(
    [0.0, -sim_vmax, -sim_vmax],
    [sim_Lx, sim_vmax, sim_vmax],
    [sim_Nx, sim_Nv, sim_Nv],
    1, 1.0, 1
)




In [ ]:
diag_par_cold = run_sim_cold(bslLD.EMSolverDKNoPol, sim_mu_em, sim_dt_cold,
                             grid_par, sim_Tmax_cold, sim_eps_cold, sim_Nx)

k_pc, ω_pc, spec_pc, nk_pc, nw_pc = make_spec_sim(diag_par_cold, sim_dt_cold, grid_par, sim_Nx)

fig = Figure(size=(900, 500))
ax = Axis(fig[1, 1],
    xlabel="k∥ ρi", ylabel="ω/Ωi",
    title="Parallel propagation: cold EMSolverDKNoPol vs QN NDC cold analytic",
    limits=((0, maximum(k_pc[2:nk_pc])), (0, 8))
)

heatmap!(ax, k_pc[2:nk_pc], ω_pc[2:nw_pc], spec_pc[2:nw_pc, 2:end]';
         colormap=:inferno)

for j in 1:size(cold_par_ref, 2)
    lines!(ax, collect(kz_values), cold_par_ref[:, j];
           color=:white, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold QN NDC analytic" : nothing))
end

scatter!(ax, kpar_k, ωre; label="hot DK analytic Re(ω)<1")


axislegend(ax; position=:lt)
fig

### Perpendicular propagation: cold simulation vs analytic

### Parallel propagation: simulation vs analytic

In [ ]:
k_par, ω_par, spec_par, nk_par, nw_par = make_spec_sim(diag_par, sim_dt, grid_par, sim_Nx)

fig = Figure(size=(900, 900))

ax1 = Axis(fig[1, 1],
    xlabel="k∥ ρi", ylabel="ω/Ωi",
    title="Parallel propagation: EMSolverDKNoPol vs QN NDC analytic",
    limits=((0, maximum(k_par[2:nk_par])), (0, 8))
)
ax2 = Axis(fig[2, 1],
    xlabel="k∥ ρi", ylabel="Im(ω)/Ωi",
    title="Kinetic analytic: Im(ω) (only roots with |Im(ω)| < $(γmax))",
    limits=((0, maximum(k_par[2:nk_par])), (-1, 1))
)

heatmap!(ax1, k_par[2:nk_par], ω_par[2:nw_par], spec_par[2:nw_par, 2:end]';
         colormap=:inferno)

for j in 1:size(cold_par_ref, 2)
    lines!(ax1, collect(kz_values), cold_par_ref[:, j];
           color=:white, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold analytic" : nothing))
end


for (iv, v) in enumerate(grid_par.vaxes[1])
    lines!(ax1, k_par[2:nk_par],k_par[2:nk_par] .* v .* ones(nk_par-1);
           label=(iv == 1 ? "v-grid" : nothing))
end


scatter!(ax1, kpar_k, ωre;
         label="hot Re(ω)")

axislegend(ax1; position=:lt)

scatter!(ax2, kpar_k[mask_red], ωim[mask_red]; color=:red)
scatter!(ax2, kpar_k[mask_blue], ωim[mask_blue]; color=:blue)
hlines!(ax2, [0.0]; color=:gray, linestyle=:dash)

fig

In [ ]:
# Bdir=1: B₀ ∥ x̂ = wave direction → k ∥ B₀ (parallel propagation)
grid_par = bslLD.Grid(
    [0.0, -sim_vmax, -sim_vmax],
    [sim_Lx, sim_vmax, sim_vmax],
    [sim_Nx, sim_Nv, sim_Nv],
    1, 1.0, 1
)
diag_par = run_sim(bslLD.EMSolverDKNoPol, sim_mu_em, sim_dt, grid_par,
                   sim_T, sim_Tmax, sim_eps, sim_Nx)

In [ ]:
k_par, ω_par, spec_par, nk_par, nw_par = make_spec_sim(diag_par, sim_dt, grid_par, sim_Nx)

fig = Figure(size=(900, 900))

ax1 = Axis(fig[1, 1],
    xlabel="k∥ ρi", ylabel="ω/Ωi",
    title="Parallel propagation: EMSolverDKNoPol vs QN NDC analytic",
    limits=((0, maximum(k_par[2:nk_par])), (0, 8))
)
ax2 = Axis(fig[2, 1],
    xlabel="k∥ ρi", ylabel="Im(ω)/Ωi",
    title="Kinetic analytic: Im(ω) (only roots with |Im(ω)| < $(γmax))",
    limits=((0, maximum(k_par[2:nk_par])), (-1, 1))
)

heatmap!(ax1, k_par[2:nk_par], ω_par[2:nw_par], spec_par[2:nw_par, 2:end]';
         colormap=:inferno)

for j in 1:size(cold_par_ref, 2)
    lines!(ax1, collect(kz_values), cold_par_ref[:, j];
           color=:white, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold analytic" : nothing))
end


for (iv, v) in enumerate(grid_par.vaxes[1])
    lines!(ax1, k_par[2:nk_par],k_par[2:nk_par] .* v .* ones(nk_par-1);
           label=(iv == 1 ? "v-grid" : nothing))
end


scatter!(ax1, kpar_k, ωre;
         label="hot Re(ω)")

axislegend(ax1; position=:lt)

scatter!(ax2, kpar_k[mask_red], ωim[mask_red]; color=:red)
scatter!(ax2, kpar_k[mask_blue], ωim[mask_blue]; color=:blue)
hlines!(ax2, [0.0]; color=:gray, linestyle=:dash)

fig

In [ ]:
fig = Figure(size=(900, 700))

ax = Axis(fig[1, 1],
    xlabel="t", ylabel="⟨|Bz|²⟩, ⟨|Ey|²⟩",
    yscale=log
)

lines!(ax, map(x -> mean(x .^ 2), diag_par.Bz))
lines!(ax, map(x -> mean(x .^ 2), diag_par.Ey))

fig

### Perpendicular propagation: simulation vs analytic

In [ ]:
# Bdir=3: B₀ ∥ ẑ, wave direction x̂ → k ⊥ B₀ (perpendicular propagation)
grid_perp = bslLD.Grid(
    [0.0, -sim_vmax, -sim_vmax],
    [sim_Lx, sim_vmax, sim_vmax],
    [sim_Nx, sim_Nv, sim_Nv],
    1, 1.0, 3
)


In [ ]:
diag_perp_cold = run_sim_cold(bslLD.EMSolverDKNoPol, sim_mu_em, sim_dt_cold,
                              grid_perp, sim_Tmax_cold, sim_eps_cold, sim_Nx)

k_pc2, ω_pc2, spec_pc2, nk_pc2, nw_pc2 = make_spec_sim(diag_perp_cold, sim_dt_cold, grid_perp, sim_Nx)

fig = Figure(size=(900, 500))
ax = Axis(fig[1, 1],
    xlabel="k⊥ ρi", ylabel="ω/Ωi",
    title="Perpendicular propagation: cold EMSolverDKNoPol vs QN NDC cold analytic",
    limits=((0, maximum(k_pc2[2:nk_pc2])), (0, 8))
)

heatmap!(ax, k_pc2[2:nk_pc2], ω_pc2[2:nw_pc2], spec_pc2[2:nw_pc2, 2:end]';
         colormap=:inferno)

for j in 1:size(cold_perp_ref, 2)
    lines!(ax, collect(kx_values), cold_perp_ref[:, j];
           color=:white, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold QN NDC analytic" : nothing))
end

scatter!(ax, first.(perp_spectrum), last.(perp_spectrum);
         color=:cyan,label="hot DK analytic")

axislegend(ax; position=:lt)
fig

In [ ]:

diag_perp = run_sim(bslLD.EMSolverDKNoPol, sim_mu_em, sim_dt, grid_perp,
                    sim_T, sim_Tmax, sim_eps, sim_Nx)


In [ ]:


k_perp, ω_perp, spec_perp, nk_perp, nw_perp = make_spec_sim(diag_perp, sim_dt, grid_perp, sim_Nx; field = :Ex)

fig = Figure(size=(900, 500))

ax = Axis(fig[1, 1],
    xlabel="k⊥ ρi", ylabel="ω/Ωi",
    title="Perpendicular propagation: EMSolverDKNoPol vs QN NDC analytic",
    limits=((0, maximum(k_perp[2:nk_perp])), (0, 8))
)

heatmap!(ax, k_perp[2:nk_perp], ω_perp[2:nw_perp], spec_perp[2:nw_perp, 2:end]';
         colormap=:inferno)

for j in 1:size(cold_perp_ref, 2)
    lines!(ax, collect(kx_values), cold_perp_ref[:, j];
           color=:white, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold analytic" : nothing))
end

scatter!(ax, first.(perp_spectrum), last.(perp_spectrum); label="hot DK analytic")

axislegend(ax; position=:lt)
fig

In [ ]:
# Bdir=3: B₀ ∥ ẑ, wave direction x̂ → k ⊥ B₀ (perpendicular propagation)
grid_perp = bslLD.Grid(
    [0.0, -sim_vmax, -sim_vmax],
    [sim_Lx, sim_vmax, sim_vmax],
    [sim_Nx, 2*sim_Nv, 2*sim_Nv],
    1, 1.0, 3
)
  

diag_perp = run_sim(bslLD.EMSolverDKNoPol, sim_mu_em, sim_dt, grid_perp,
                    sim_T, sim_Tmax, sim_eps, sim_Nx)


k_perp, ω_perp, spec_perp, nk_perp, nw_perp = make_spec_sim(diag_perp, sim_dt, grid_perp, sim_Nx; field = :Ex)

fig = Figure(size=(900, 500))

ax = Axis(fig[1, 1],
    xlabel="k⊥ ρi", ylabel="ω/Ωi",
    title="Perpendicular propagation: EMSolverDKNoPol vs QN NDC analytic",
    limits=((0, maximum(k_perp[2:nk_perp])), (0, 8))
)

heatmap!(ax, k_perp[2:nk_perp], ω_perp[2:nw_perp], spec_perp[2:nw_perp, 2:end]';
         colormap=:inferno)

for j in 1:size(cold_perp_ref, 2)
    lines!(ax, collect(kx_values), cold_perp_ref[:, j];
           color=:white, linestyle=:dash, linewidth=1.5,
           label=(j == 1 ? "cold analytic" : nothing))
end

scatter!(ax, first.(perp_spectrum), last.(perp_spectrum); label="hot DK analytic")

axislegend(ax; position=:lt)
fig

In [ ]:
fig = Figure(size=(900, 700))

ax = Axis(fig[1, 1],
    xlabel="t", ylabel="⟨|Bz|²⟩, ⟨|Ey|²⟩",
    yscale=log
)

lines!(ax, map(x -> mean(x .^ 2), diag_perp.Bz))
lines!(ax, map(x -> mean(x .^ 2), diag_perp.Ex))
lines!(ax, map(x -> mean(x .^ 2), diag_perp.Ey))

fig